# 01 · Calibración del piso de ruido — panel de 30

Este notebook mide **cuánto se mueve el pipeline solo por azar**, usando únicamente
las corridas sobre los 30 videos. Nada de lo que sale acá depende del panel de 14
ni de `reports/noise_floor.json`.

Es el **paso 1 de dos**: acá se fija el umbral, ciego a las hipótesis. El notebook
`02_evidencia_ablacion.ipynb` usa esos umbrales para juzgar los experimentos. Esa
separación importa — si el mismo notebook calculara el ruido *y* evaluara las
hipótesis, se podría ajustar el umbral hasta que dé el veredicto deseado.

## Qué es una réplica nula

Dos corridas son una *réplica nula* si difieren **solo** en la llamada al modelo:
mismo prompt de Stage 2 (verificado por sha256), mismo World Model de entrada,
misma pizarra, mismo transcript, mismo evaluador.

Bajo esas condiciones la diferencia verdadera entre ambas es **cero por construcción**.
Todo lo que se mida es ruido.

> Temperatura 0.0 **no** garantiza determinismo: fija el muestreo, pero el batching
> y el orden de reducción en GPU varían entre llamadas.

## Orden de las celdas

1. Inventario de corridas del panel de 30
2. Qué pares califican como réplica nula (y cuáles se descartan, con el motivo)
3. Tabla por video — Edge F1
4. Tabla por video — Service F1
5. **Piso de ruido de cada métrica** — no solo F1
6. Cómo se calculan sigma_d y el MDE, paso a paso
7. Exportación de las constantes para el notebook 02


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ABL = Path("../reports/ablation")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)


def load_run(dirname: str) -> dict:
    return json.loads((ABL / dirname / "run.json").read_text(encoding="utf-8"))


def scores(run: dict) -> dict:
    """{video_id: (svc, edge)} en escala 0-100, solo los videos exitosos.

    El runner guarda las metricas por video en 0-1; las corridas ensambladas a mano
    las guardaron en 0-100. Se normaliza mirando el rango, no el nombre del archivo.
    """
    out = {}
    for r in run["results"]:
        if r.get("status") not in (None, "success"):
            continue
        # results/ablation/ tambien guarda corridas de otra naturaleza (p. ej. la
        # auditoria de aristas de retorno, con campos draft_/audited_). No son
        # corridas de ablacion y no tienen con que puntuarse aca.
        if "svc_f1" not in r or "edge_f1" not in r:
            continue
        svc, edge = r["svc_f1"], r["edge_f1"]
        if max(svc, edge) <= 1.0000001:
            svc, edge = svc * 100, edge * 100
        out[r["video_id"]] = (svc, edge)
    return out


# Inventario de todo lo que existe sobre 30 videos
rows = []
for p in sorted(ABL.glob("*/run.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    if "stage2_prompt" not in d:
        continue          # no es una corrida de ablacion
    s = scores(d)
    if len(s) < 30:
        continue
    orc = d.get("oracle") or {}
    rows.append({
        "corrida": p.parent.name,
        "n": len(s),
        "prompt": d["stage2_prompt"]["name"],
        "sha": d["stage2_prompt"]["sha256"][:10],
        "oraculo": orc.get("name"),
        "connev": bool(d.get("connection_evidence_enabled")),
        # La ablacion de transcript corre con el MISMO prompt y sin oraculo, asi que
        # sin este campo se confundiria con una replica nula de produccion — y el piso
        # de ruido terminaria midiendo un efecto real como si fuera azar.
        "transcript": d.get("transcript_enabled", True),
        "replicate": d.get("replicate"),
        "ensamblada": any("source" in r for r in d["results"]),
    })

inventario = pd.DataFrame(rows)
print(f"Corridas con los 30 videos: {len(inventario)}\n")
inventario

Corridas con los 30 videos: 21



,corrida,n,prompt,sha,oraculo,connev,transcript,replicate,ensamblada
0,2026-08-24_0501_STAGE2_V6_CORRECTED_cell9_panel30,30,STAGE2_V6_CORRECTED,ed1d85054d,NaN,False,True,NaN,True
1,2026-08-25_0207_STAGE2_V4_ANTI_HALLUCINATION_c...,30,STAGE2_V4_ANTI_HALLUCINATION,cdb8998f48,NaN,False,True,NaN,False
2,2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30,30,STAGE2_V6_CORRECTED,ed1d85054d,NaN,False,True,NaN,False
3,2026-08-25_1708_STAGE2_V5_STRICT_ROUTING_cell7...,30,STAGE2_V5_STRICT_ROUTING,53f8d91244,NaN,False,True,NaN,False
4,2026-08-25_1737_STAGE2_V7_RETURN_FLOWS_cell7_p30,30,STAGE2_V7_RETURN_FLOWS,1ed4ebf91e,NaN,False,True,NaN,False
5,2026-08-26_0135_STAGE2_V6_CORRECTED_cell9_p30_...,30,STAGE2_V6_CORRECTED,ed1d85054d,NaN,True,True,NaN,False
6,2026-08-26_0307_STAGE2_V8_ACTORS_AND_RETURNS_c...,30,STAGE2_V8_ACTORS_AND_RETURNS,277020bacd,NaN,False,True,NaN,False
7,2026-08-26_2132_STAGE2_V5_STRICT_ROUTING_cell8...,30,STAGE2_V5_STRICT_ROUTING,a792c1328d,NaN,False,True,NaN,False
8,2026-08-26_2155_STAGE2_V6_OPTIMIZED_cell8_p30,30,STAGE2_V6_OPTIMIZED,079b0aa855,NaN,False,True,NaN,False
9,2026-08-26_2220_STAGE2_V7_RETURN_FLOWS_V6_cell...,30,STAGE2_V7_RETURN_FLOWS_V6,7ad8d9368b,NaN,False,True,NaN,False


## 2 · Qué pares califican como réplica nula

Para que un par mida **solo ruido**, las dos corridas tienen que compartir prompt
(mismo sha256) **y** condición de entrada. Tres cosas rompen eso:

| Se descarta si… | Por qué |
|---|---|
| `connev` distinto | agrega evidencia de conexiones al contexto — es otra entrada |
| `oraculo` distinto | reemplaza el World Model de Stage 1 — es *el efecto que estamos midiendo*, no ruido |
| `ensamblada = True` | sus resultados fueron **copiados** de otras corridas, no generados de nuevo |

El tercer caso es el que más engaña. `2026-08-24_0501_..._panel30` parece una tercera
réplica de producción sobre los 30, pero cada uno de sus resultados trae un campo
`source` que apunta a la corrida original: sus 14 primeros videos son una copia literal
de `2026-08-22_0332`. Emparejarla mediría 0 de ruido en esos 14 y contaminaría el sigma
hacia abajo.

In [2]:
def firma_condicion(r) -> tuple:
    """Dos corridas son comparables como replica nula si esto coincide.

    `oraculo` viene como NaN cuando no hay oraculo, y NaN es *truthy* en Python:
    usarlo directo en un `if` etiquetaria toda corrida de produccion como oraculo.
    Se normaliza a None antes de comparar.

    `transcript` entra en la firma porque la ablacion de transcript comparte prompt y
    ausencia de oraculo con produccion: sin ese campo, `oracle-vision` y
    `oracle-vision_notranscript` se emparejaban como replica nula e inflaban sigma_d.
    """
    orc = r["oraculo"] if isinstance(r["oraculo"], str) else None
    return (r["sha"], orc, bool(r["connev"]), bool(r["transcript"]))


candidatas = inventario[~inventario["ensamblada"]].copy()
descartadas = inventario[inventario["ensamblada"]].copy()

pares, solitarias, sobrantes = [], [], []
for firma, grupo in candidatas.groupby(candidatas.apply(firma_condicion, axis=1)):
    nombres = sorted(grupo["corrida"])
    if len(nombres) < 2:
        solitarias.append((firma, nombres[0]))
        continue
    # Con 2 corridas hay 1 par posible. Con 3+ (ej. rep1/rep2/rep3 de produccion),
    # tomar TODAS las combinaciones reutiliza cada corrida en varios pares y rompe
    # la independencia que el resto del notebook exige (una diferencia pareada no
    # puede depender de la misma corrida dos veces). Se empareja consecutivo — (0,1),
    # (2,3), ... — y la que sobra por ser cantidad impar queda afuera, no descartada
    # del analisis en general, solo de esta muestra de sigma_d.
    for i in range(0, len(nombres) - 1, 2):
        pares.append({"condicion": "oraculo" if firma[1] else "produccion",
                      "sha": firma[0], "run_a": nombres[i], "run_b": nombres[i + 1]})
    if len(nombres) % 2:
        sobrantes.append((firma, nombres[-1]))

print(f"PARES VALIDOS: {len(pares)}")
for p in pares:
    print(f"  [{p['condicion']}] {p['run_a']}\n           vs {p['run_b']}")

print(f"\nDESCARTADAS por ensambladas: {len(descartadas)}")
for _, r in descartadas.iterrows():
    print(f"  {r['corrida']} — resultados copiados de otras corridas")

print(f"\nSIN PAREJA (una sola corrida en su condicion): {len(solitarias)}")
for firma, nombre in solitarias:
    print(f"  {nombre}  (sha={firma[0]}, oraculo={firma[1]}, connev={firma[2]})")

if sobrantes:
    print(f"\nSOBRANTE por cantidad impar en su grupo (no entra en ningun par, "
          f"para no reusar una corrida): {len(sobrantes)}")
    for firma, nombre in sobrantes:
        print(f"  {nombre}  (sha={firma[0]}, oraculo={firma[1]}, connev={firma[2]})")

PARES VALIDOS: 3
  [oraculo] 2026-08-28_0315_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt
           vs 2026-08-28_2211_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt_rep2
  [produccion] 2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30
           vs 2026-08-27_1640_STAGE2_V6_CORRECTED_cell9_p30_rep2
  [produccion] 2026-08-29_0442_STAGE2_V6_CORRECTED_cell9_p30_rep3
           vs 2026-08-29_0715_STAGE2_V6_CORRECTED_cell9_p30_rep4

DESCARTADAS por ensambladas: 1
  2026-08-24_0501_STAGE2_V6_CORRECTED_cell9_panel30 — resultados copiados de otras corridas

SIN PAREJA (una sola corrida en su condicion): 14
  2026-08-26_2155_STAGE2_V6_OPTIMIZED_cell8_p30  (sha=079b0aa855, oraculo=None, connev=False)
  2026-08-25_1737_STAGE2_V7_RETURN_FLOWS_cell7_p30  (sha=1ed4ebf91e, oraculo=None, connev=False)
  2026-08-26_0307_STAGE2_V8_ACTORS_AND_RETURNS_cellNone_p30  (sha=277020bacd, oraculo=None, connev=False)
  2026-08-30_1733_STAGE2_V4_DYNAMIC_FEW_SHOT_cellNone_p30  (sha=45da674e44, oraculo=None, connev=

## 3 · Tabla por video — Edge F1

Cada fila es un video. `dif` es corrida A menos corrida B: **debería ser 0** si el
modelo fuera determinista. Todo lo que se aleja de 0 es ruido.

In [3]:
def tabla_par(par: dict, metrica: str) -> pd.DataFrame:
    idx = 0 if metrica == "svc" else 1
    A, B = scores(load_run(par["run_a"])), scores(load_run(par["run_b"]))
    comunes = sorted(set(A) & set(B))
    df = pd.DataFrame({
        "video": comunes,
        "corrida_A": [A[v][idx] for v in comunes],
        "corrida_B": [B[v][idx] for v in comunes],
    })
    df["dif"] = df["corrida_A"] - df["corrida_B"]
    df["|dif|"] = df["dif"].abs()
    df.insert(0, "condicion", par["condicion"])
    return df


for par in pares:
    df = tabla_par(par, "edge")
    print(f"\n{'=' * 78}")
    print(f"[{par['condicion'].upper()}]  A = {par['run_a']}")
    print(f"{' ' * (len(par['condicion']) + 3)}  B = {par['run_b']}")
    print(f"{'=' * 78}")
    display(df.round(2))
    print(f"media A = {df['corrida_A'].mean():.2f}   media B = {df['corrida_B'].mean():.2f}   "
          f"dif de medias = {df['dif'].mean():+.2f}")
    print(f"|dif| por video: media {df['|dif|'].mean():.2f} · mediana {df['|dif|'].median():.2f} "
          f"· max {df['|dif|'].max():.2f}")
    print(f"videos con dif exactamente 0: {(df['dif'] == 0).sum()}/{len(df)}")


[ORACULO]  A = 2026-08-28_0315_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt
            B = 2026-08-28_2211_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt_rep2


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,oraculo,-3lnf5lzsH0,93.33,93.33,0.00,0.00
1,oraculo,-kA0ahrhX3I,94.12,100.00,-5.88,5.88
2,oraculo,-wLEkq21cvA,85.71,90.00,-4.29,4.29
3,oraculo,07lfvavMdfU,92.31,88.89,3.42,3.42
4,oraculo,1aYoIZvabbk,90.91,90.91,0.00,0.00
5,oraculo,2L0m28ZLmtE,95.24,83.33,11.90,11.90
6,oraculo,2XVgpMwY5iE,85.71,90.91,-5.19,5.19
7,oraculo,2e3vOxsHekE,100.00,100.00,0.00,0.00
8,oraculo,6CgqEzyWpeA,85.71,91.89,-6.18,6.18
9,oraculo,6EUknQqaV1w,90.91,86.96,3.95,3.95


media A = 83.18   media B = 81.16   dif de medias = +2.02
|dif| por video: media 4.50 · mediana 4.70 · max 22.46
videos con dif exactamente 0: 10/30

[PRODUCCION]  A = 2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30
               B = 2026-08-27_1640_STAGE2_V6_CORRECTED_cell9_p30_rep2


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,produccion,-3lnf5lzsH0,32.00,32.00,0.00,0.00
1,produccion,-kA0ahrhX3I,53.33,66.67,-13.33,13.33
2,produccion,-wLEkq21cvA,77.78,50.00,27.78,27.78
3,produccion,07lfvavMdfU,75.00,69.57,5.43,5.43
4,produccion,1aYoIZvabbk,83.33,83.33,0.00,0.00
5,produccion,2L0m28ZLmtE,80.00,64.00,16.00,16.00
6,produccion,2XVgpMwY5iE,45.45,43.48,1.98,1.98
7,produccion,2e3vOxsHekE,72.73,83.33,-10.61,10.61
8,produccion,6CgqEzyWpeA,74.29,74.29,0.00,0.00
9,produccion,6EUknQqaV1w,78.26,78.26,0.00,0.00


media A = 58.99   media B = 57.86   dif de medias = +1.12
|dif| por video: media 3.92 · mediana 0.00 · max 27.78
videos con dif exactamente 0: 18/30

[PRODUCCION]  A = 2026-08-29_0442_STAGE2_V6_CORRECTED_cell9_p30_rep3
               B = 2026-08-29_0715_STAGE2_V6_CORRECTED_cell9_p30_rep4


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,produccion,-3lnf5lzsH0,32.00,32.00,0.00,0.00
1,produccion,-kA0ahrhX3I,61.54,66.67,-5.13,5.13
2,produccion,-wLEkq21cvA,73.68,77.78,-4.09,4.09
3,produccion,07lfvavMdfU,75.00,75.00,0.00,0.00
4,produccion,1aYoIZvabbk,66.67,66.67,0.00,0.00
5,produccion,2L0m28ZLmtE,66.67,64.00,2.67,2.67
6,produccion,2XVgpMwY5iE,45.45,47.62,-2.16,2.16
7,produccion,2e3vOxsHekE,83.33,72.73,10.61,10.61
8,produccion,6CgqEzyWpeA,74.29,74.29,0.00,0.00
9,produccion,6EUknQqaV1w,83.33,78.26,5.07,5.07


media A = 57.81   media B = 57.87   dif de medias = -0.06
|dif| por video: media 2.77 · mediana 0.62 · max 15.17
videos con dif exactamente 0: 15/30


### Lectura de esta tabla

Hay que mirar **dos cosas distintas**, y confundirlas es el error clásico:

- **`dif de medias`** (el promedio del panel entero) es chica — típicamente ~1 punto.
  Es lo que uno reporta como "el resultado de la corrida".
- **`|dif| por video`** es bastante más grande. Los videos individuales se mueven
  mucho entre corridas; al promediar 30, esos movimientos se cancelan entre sí.

El promedio del panel es estable **porque** promedia, no porque el modelo sea estable.
Y para saber si una diferencia entre dos experimentos es real, lo que manda es la
dispersión por video, no lo estable que se vea el promedio.

## 4 · Tabla por video — Service F1

In [4]:
for par in pares:
    df = tabla_par(par, "svc")
    print(f"\n{'=' * 78}")
    print(f"[{par['condicion'].upper()}]  A = {par['run_a']}")
    print(f"{' ' * (len(par['condicion']) + 3)}  B = {par['run_b']}")
    print(f"{'=' * 78}")
    display(df.round(2))
    print(f"media A = {df['corrida_A'].mean():.2f}   media B = {df['corrida_B'].mean():.2f}   "
          f"dif de medias = {df['dif'].mean():+.2f}")
    print(f"|dif| por video: media {df['|dif|'].mean():.2f} · mediana {df['|dif|'].median():.2f} "
          f"· max {df['|dif|'].max():.2f}")
    print(f"videos con dif exactamente 0: {(df['dif'] == 0).sum()}/{len(df)}")


[ORACULO]  A = 2026-08-28_0315_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt
            B = 2026-08-28_2211_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt_rep2


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,oraculo,-3lnf5lzsH0,100.00,100.00,0.00,0.00
1,oraculo,-kA0ahrhX3I,100.00,92.31,7.69,7.69
2,oraculo,-wLEkq21cvA,90.91,100.00,-9.09,9.09
3,oraculo,07lfvavMdfU,100.00,100.00,0.00,0.00
4,oraculo,1aYoIZvabbk,100.00,100.00,0.00,0.00
5,oraculo,2L0m28ZLmtE,100.00,100.00,0.00,0.00
6,oraculo,2XVgpMwY5iE,100.00,100.00,0.00,0.00
7,oraculo,2e3vOxsHekE,100.00,100.00,0.00,0.00
8,oraculo,6CgqEzyWpeA,100.00,100.00,0.00,0.00
9,oraculo,6EUknQqaV1w,100.00,100.00,0.00,0.00


media A = 99.70   media B = 99.44   dif de medias = +0.26
|dif| por video: media 0.86 · mediana 0.00 · max 9.09
videos con dif exactamente 0: 27/30

[PRODUCCION]  A = 2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30
               B = 2026-08-27_1640_STAGE2_V6_CORRECTED_cell9_p30_rep2


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,produccion,-3lnf5lzsH0,70.00,70.00,0.00,0.00
1,produccion,-kA0ahrhX3I,92.31,92.31,0.00,0.00
2,produccion,-wLEkq21cvA,88.89,72.73,16.16,16.16
3,produccion,07lfvavMdfU,94.74,94.74,0.00,0.00
4,produccion,1aYoIZvabbk,92.31,92.31,0.00,0.00
5,produccion,2L0m28ZLmtE,96.00,96.00,0.00,0.00
6,produccion,2XVgpMwY5iE,71.43,71.43,0.00,0.00
7,produccion,2e3vOxsHekE,83.33,83.33,0.00,0.00
8,produccion,6CgqEzyWpeA,94.74,94.74,0.00,0.00
9,produccion,6EUknQqaV1w,85.71,85.71,0.00,0.00


media A = 88.07   media B = 87.03   dif de medias = +1.03
|dif| por video: media 1.34 · mediana 0.00 · max 16.16
videos con dif exactamente 0: 26/30

[PRODUCCION]  A = 2026-08-29_0442_STAGE2_V6_CORRECTED_cell9_p30_rep3
               B = 2026-08-29_0715_STAGE2_V6_CORRECTED_cell9_p30_rep4


,condicion,video,corrida_A,corrida_B,dif,|dif|
0,produccion,-3lnf5lzsH0,70.00,70.00,0.00,0.00
1,produccion,-kA0ahrhX3I,92.31,92.31,0.00,0.00
2,produccion,-wLEkq21cvA,88.89,88.89,0.00,0.00
3,produccion,07lfvavMdfU,94.74,94.74,0.00,0.00
4,produccion,1aYoIZvabbk,76.92,76.92,0.00,0.00
5,produccion,2L0m28ZLmtE,100.00,96.00,4.00,4.00
6,produccion,2XVgpMwY5iE,71.43,71.43,0.00,0.00
7,produccion,2e3vOxsHekE,83.33,83.33,0.00,0.00
8,produccion,6CgqEzyWpeA,94.74,94.74,0.00,0.00
9,produccion,6EUknQqaV1w,85.71,85.71,0.00,0.00


media A = 87.19   media B = 86.13   dif de medias = +1.06
|dif| por video: media 1.18 · mediana 0.00 · max 16.67
videos con dif exactamente 0: 25/30


## 5 · Cómo se calculan sigma_d y el MDE

### sigma_d — la dispersión de la diferencia pareada

Para cada video $i$ tomamos la diferencia entre las dos corridas:

$$d_i = \text{F1}_i^{(A)} - \text{F1}_i^{(B)}$$

Como las dos corridas son idénticas en todo salvo la llamada al modelo, el valor
esperado de $d_i$ es 0. `sigma_d` es el desvío estándar muestral de esas diferencias:

$$\sigma_d = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(d_i - \bar{d})^2}$$

Se usa `ddof=1` (divisor $n-1$, no $n$) porque estimamos la media a partir de la
propia muestra.

**Es dispersión *por video*, no del promedio del panel.** Un `sigma_d` de 7 no
significa que el promedio del panel se mueva 7 puntos — significa que un video
cualquiera se mueve típicamente ~7 puntos entre corridas.

### MDE — efecto mínimo detectable

La diferencia más chica que un panel de $n$ videos puede distinguir del ruido, con
80 % de potencia y 5 % de significancia, en un test pareado a dos colas:

$$\text{MDE}(n) = (z_{1-\alpha/2} + z_{1-\beta}) \cdot \frac{\sigma_d}{\sqrt{n}}
= (1.96 + 0.84) \cdot \frac{\sigma_d}{\sqrt{n}} = 2.80 \cdot \frac{\sigma_d}{\sqrt{n}}$$

El $\sqrt{n}$ es el motivo por el que el promedio del panel es mucho más estable que
un video suelto: promediar 30 videos divide el ruido por $\sqrt{30} \approx 5.5$.

**Cómo se usa:** si dos experimentos difieren en menos que el MDE, esa diferencia no
es distinguible del azar con este tamaño de panel — sin importar lo convincente que
se vea la tabla.

In [5]:
Z_MDE = 1.96 + 0.84  # dos colas, alfa=0.05, potencia=0.80

difs = {"svc": [], "edge": []}
detalle = []

for par in pares:
    fila = {"condicion": par["condicion"]}
    for metrica in ("svc", "edge"):
        d = tabla_par(par, metrica)["dif"].to_numpy()
        difs[metrica].extend(d.tolist())
        fila[f"n"] = len(d)
        fila[f"sigma_{metrica}"] = d.std(ddof=1)
    detalle.append(fila)

print("sigma_d por par, antes de juntar:")
display(pd.DataFrame(detalle).round(2))

# Los pares no comparten ninguna corrida entre si, asi que sus diferencias son
# observaciones independientes y se pueden juntar en una sola muestra.
usadas = [p["run_a"] for p in pares] + [p["run_b"] for p in pares]
assert len(usadas) == len(set(usadas)), "Hay una corrida repetida en dos pares: no son independientes"

resumen = []
for metrica, etiqueta in (("svc", "Service F1"), ("edge", "Edge F1")):
    d = np.array(difs[metrica])
    sd = d.std(ddof=1)
    resumen.append({
        "metrica": etiqueta,
        "n observaciones": len(d),
        "sigma_d por video": sd,
        "MDE n=14": Z_MDE * sd / np.sqrt(14),
        "MDE n=30": Z_MDE * sd / np.sqrt(30),
        "MDE n=100": Z_MDE * sd / np.sqrt(100),
        "n para 3 pts": int(np.ceil((Z_MDE * sd / 3.0) ** 2)),
    })

print(f"\nPOOL de {len(pares)} pares independientes ({len(difs['edge'])} observaciones)")
resumen_df = pd.DataFrame(resumen).set_index("metrica")
display(resumen_df.round(2))

sigma_d por par, antes de juntar:


,condicion,n,sigma_svc,sigma_edge
0,oraculo,30,2.77,6.31
1,produccion,30,4.06,7.84
2,produccion,30,3.46,4.84



POOL de 3 pares independientes (90 observaciones)


,n observaciones,sigma_d por video,MDE n=14,MDE n=30,MDE n=100,n para 3 pts
metrica,,,,,,
Service F1,90,3.45,2.58,1.76,0.97,11
Edge F1,90,6.43,4.81,3.29,1.80,37


## 6 · El piso de ruido de **cada** métrica

Este es el punto que suele saltearse. **Cada cantidad medida tiene su propio piso de
ruido**, y no son intercambiables: no se puede juzgar una diferencia en *cantidad de
aristas* contra el MDE del *Edge F1*. Son unidades distintas y dispersiones distintas.

Si el notebook 02 va a afirmar algo sobre precisión, sobre cuántas aristas dibuja un
prompt o sobre cuántos servicios alucina, el umbral de esa afirmación tiene que salir
de acá — calculado sobre las mismas réplicas nulas.

Las cantidades que ya vienen por video en cada `run.json`:

| métrica | qué mide |
|---|---|
| `edge_f1`, `svc_f1` | calidad contra el ground truth |
| `gen_edges` | cuántas aristas dibujó el modelo (vs `gt_edges` reales) |
| `gen_nodes` | cuántos nodos dibujó |
| `services_hallucinated` | servicios inventados que no están en el GT |
| `services_missing` | servicios del GT que no detectó |

In [6]:
METRICAS = {
    "Edge F1":            lambda r: 100 * r["edge_f1"],
    "Service F1":         lambda r: 100 * r["svc_f1"],
    "aristas generadas":  lambda r: r["gen_edges"],
    "nodos generados":    lambda r: r["gen_nodes"],
    "svc alucinados":     lambda r: len(r.get("services_hallucinated") or []),
    "svc faltantes":      lambda r: len(r.get("services_missing") or []),
}


def difs_por_metrica(pares: list[dict]) -> dict[str, np.ndarray]:
    """Diferencias pareadas por video, para cada metrica, juntando todos los pares."""
    out = {k: [] for k in METRICAS}
    for par in pares:
        A, B = scores_raw(par["run_a"]), scores_raw(par["run_b"])
        for v in sorted(set(A) & set(B)):
            for nombre, f in METRICAS.items():
                out[nombre].append(f(A[v]) - f(B[v]))
    return {k: np.array(v) for k, v in out.items()}


def scores_raw(dirname: str) -> dict:
    """{video_id: result} crudo, para poder leer cualquier campo, no solo F1."""
    return {r["video_id"]: r for r in load_run(dirname)["results"]
            if r.get("status") == "success"}


todas = difs_por_metrica(pares)

filas = []
for nombre, d in todas.items():
    sd = d.std(ddof=1)
    filas.append({
        "metrica": nombre,
        "n obs": len(d),
        "sigma_d": sd,
        "MDE n=14": Z_MDE * sd / np.sqrt(14),
        "MDE n=30": Z_MDE * sd / np.sqrt(30),
        "media |dif|": np.abs(d).mean(),
        "max |dif|": np.abs(d).max(),
    })

piso = pd.DataFrame(filas).set_index("metrica")
print("Piso de ruido por metrica — pool de los pares nulos del panel de 30\n")
display(piso.round(2))

Piso de ruido por metrica — pool de los pares nulos del panel de 30



,n obs,sigma_d,MDE n=14,MDE n=30,media |dif|,max |dif|
metrica,,,,,,
Edge F1,90,6.43,4.81,3.29,3.73,27.78
Service F1,90,3.45,2.58,1.76,1.13,16.67
aristas generadas,90,1.42,1.06,0.73,0.82,6.00
nodos generados,90,0.66,0.50,0.34,0.21,4.00
svc alucinados,90,0.41,0.31,0.21,0.13,2.00
svc faltantes,90,0.21,0.16,0.11,0.04,1.00


### Cómo leer esta tabla

`MDE n=30` es **la barra que hay que superar** para afirmar algo con ese panel.

Concretamente: si dos prompts difieren en 1.2 aristas generadas en promedio y el MDE
de esa métrica es 0.79, esa diferencia **sí** es real aunque sus Edge F1 sean
indistinguibles. Usar solo F1 haría invisible ese hallazgo.

## 7 · El piso de ruido del protocolo **permisivo**

El protocolo permisivo puntúa **los mismos grafos ya generados** con un criterio más
laxo. No hay llamadas nuevas a la API: cambia el evaluador, no la salida del modelo.

Es un **instrumento de medición distinto**, así que necesita su propio piso de ruido —
la misma regla que aplicamos a las otras métricas. Un MDE calculado sobre el protocolo
estricto no sirve para juzgar una diferencia medida en permisivo.

### El permisivo relaja tres cosas a la vez

`build_evidence_tables.py:eval_permissive` afloja simultáneamente:

1. **Colapso de subtipos de actor** — todo `User*` → `User`, todo `ThirdParty*` → `ThirdParty`
2. **Aristas como conjunto** — las instancias duplicadas del mismo par dejan de contar aparte
3. **Aristas no dirigidas** — `A → B` y `B → A` se vuelven la misma arista

Vale la pena **separarlas**. Lo que pediste —agrupar users y third parties— es solo la
relajación 1, y es la más defendible: dice que confundir `UserConsumerWeb` con
`UserCompanyDeveloper` es una discusión de nomenclatura, no un error estructural.

La 3 es mucho más agresiva: dice que **equivocarse en la dirección de la flecha no
penaliza**. Dado que el pipeline entero trata de reconstruir el flujo, eso descarta
señal real. En la Tabla 2 del paper el recall de aristas sube +24 puntos con el
permisivo completo, y hace falta saber cuánto de eso viene de cada nivel antes de
citarlo.

Por eso acá se calibran **cuatro niveles acumulativos**, no uno.

In [7]:
import csv
from collections import Counter

from rich.console import Console

import networkx as nx

import sys
sys.path.append("..")
import scripts.core.graph_builder as _gb
from scripts.core.graph_builder import create_graph_from_cloudscape_json

# graph_builder escribe una linea por grafo con su propia consola `rich`, que toma
# sys.stdout al importarse — redirect_stdout no la alcanza. Con 4 niveles x 6 corridas
# x 30 videos serian ~720 lineas tapando las tablas, asi que se silencia en el origen.
_gb.console = Console(quiet=True)

GT_DIR = Path("../data/cloudscape_gt")
CATALOG = {r["name"]: r for r in csv.DictReader(open("../data/cloudscape_gt/services.csv", encoding="utf-8"))}


def norm_actor(svc: str) -> str:
    """Colapsa subtipos de actor. Copia fiel de build_evidence_tables.norm_permissive."""
    if not svc or svc == "?":
        return "Unknown"
    s = str(svc).strip()
    cap = CATALOG.get(s, {}).get("capability", "")
    if s.startswith("User") or cap == "User":
        return "User"
    if s.startswith("ThirdParty") or cap == "ThirdParty":
        return "ThirdParty"
    return s


def _prf(inter, gen, gt):
    p = inter / gen if gen else 0.0
    r = inter / gt if gt else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0


def puntuar(G_gen, G_gt, nivel: int) -> tuple[float, float]:
    """Svc F1 y Edge F1 bajo un nivel acumulativo de permisividad.

    0 = estricto        servicios exactos, aristas multiconjunto dirigido
    1 = + actores       User*/ThirdParty* colapsados
    2 = + sin duplicar  aristas como conjunto
    3 = + sin direccion aristas no dirigidas   (= el permisivo completo del paper)
    """
    norm = norm_actor if nivel >= 1 else (lambda s: s or "?")

    gen_s = [norm(G_gen.nodes[n].get("service", "")) for n in G_gen]
    gt_s = [norm(G_gt.nodes[n].get("service", "")) for n in G_gt]
    if nivel >= 1:
        # El permisivo del paper compara servicios por CONJUNTO, no multiconjunto.
        gi, gg, tt = len(set(gen_s) & set(gt_s)), len(set(gen_s)), len(set(gt_s))
    else:
        ca, cb = Counter(gen_s), Counter(gt_s)
        gi, gg, tt = sum((ca & cb).values()), sum(ca.values()), sum(cb.values())
    svc = _prf(gi, gg, tt)

    def aristas(G):
        out = []
        for u, v in G.edges():
            a, b = norm(G.nodes[u].get("service", "")), norm(G.nodes[v].get("service", ""))
            if nivel >= 1 and (a == "Unknown" or b == "Unknown"):
                continue
            out.append(tuple(sorted([a, b])) if nivel >= 3 else (a, b))
        return set(out) if nivel >= 2 else out

    ge, gt_e = aristas(G_gen), aristas(G_gt)
    if nivel >= 2:
        ei, eg, et = len(ge & gt_e), len(ge), len(gt_e)
    else:
        ca, cb = Counter(ge), Counter(gt_e)
        ei, eg, et = sum((ca & cb).values()), sum(ca.values()), sum(cb.values())
    return svc * 100, _prf(ei, eg, et) * 100


_gt_cache: dict[str, nx.DiGraph] = {}


def gt_graph(vid: str) -> nx.DiGraph:
    if vid not in _gt_cache:
        _gt_cache[vid] = nx.read_graphml(str(GT_DIR / f"{vid}.graphml"))
    return _gt_cache[vid]


def puntajes_nivel(dirname: str, nivel: int) -> dict[str, tuple[float, float]]:
    """Re-puntua una corrida entera desde los grafos crudos guardados en run.json.

    `create_graph_from_cloudscape_json` imprime una linea por grafo; con 4 niveles x
    6 corridas x 30 videos eso son 720 lineas de ruido que tapan la tabla.
    """
    out = {}
    for r in load_run(dirname)["results"]:
        if r.get("status") != "success" or not r.get("analysis"):
            continue
        vid = r["video_id"]
        G = create_graph_from_cloudscape_json(r["analysis"], video_id=vid)
        out[vid] = puntuar(G, gt_graph(vid), nivel)
    return out


NIVELES = {0: "0 · estricto", 1: "1 · + actores agrupados",
           2: "2 · + aristas sin duplicar", 3: "3 · + aristas sin dirección"}

filas = []
for nivel, etiqueta in NIVELES.items():
    d_svc, d_edge, medias_svc, medias_edge = [], [], [], []
    for par in pares:
        A = puntajes_nivel(par["run_a"], nivel)
        B = puntajes_nivel(par["run_b"], nivel)
        for v in sorted(set(A) & set(B)):
            d_svc.append(A[v][0] - B[v][0])
            d_edge.append(A[v][1] - B[v][1])
            medias_svc += [A[v][0], B[v][0]]
            medias_edge += [A[v][1], B[v][1]]
    sd_s, sd_e = np.std(d_svc, ddof=1), np.std(d_edge, ddof=1)
    filas.append({
        "nivel": etiqueta,
        "Svc F1 medio": np.mean(medias_svc), "sigma_d Svc": sd_s,
        "MDE Svc n=30": Z_MDE * sd_s / np.sqrt(30),
        "Edge F1 medio": np.mean(medias_edge), "sigma_d Edge": sd_e,
        "MDE Edge n=30": Z_MDE * sd_e / np.sqrt(30),
    })

permisivo = pd.DataFrame(filas).set_index("nivel")
print(f"Calibrado sobre los mismos {len(pares)} pares nulos "
      f"({len(d_edge)} diferencias por nivel)\n")
display(permisivo.round(2))

Calibrado sobre los mismos 3 pares nulos (90 diferencias por nivel)



,Svc F1 medio,sigma_d Svc,MDE Svc n=30,Edge F1 medio,sigma_d Edge,MDE Edge n=30
nivel,,,,,,
0 · estricto,89.50,3.58,1.83,66.15,6.43,3.29
1 · + actores agrupados,94.38,3.51,1.79,68.18,6.61,3.38
2 · + aristas sin duplicar,94.38,3.51,1.79,71.47,6.03,3.08
3 · + aristas sin dirección,94.38,3.51,1.79,82.44,5.57,2.85


### Cómo leer esta tabla

Dos columnas distintas que no hay que confundir:

- **`F1 medio`** sube al relajar — es el efecto buscado: el criterio perdona más.
- **`sigma_d` / `MDE`** es lo que importa acá: si el permisivo *también* reduce el
  ruido, entonces detecta diferencias más chicas. Si el ruido baja menos que lo que
  sube el F1, la permisividad está inflando el puntaje sin mejorar la capacidad de
  distinguir condiciones.

La pregunta que responde: **¿el permisivo mide mejor, o solo puntúa más alto?**

In [8]:
# Cuanto aporta cada relajacion por separado, sobre el nivel anterior
delta = permisivo.diff().iloc[1:]
delta.index = ["+ agrupar actores", "+ aristas sin duplicar", "+ aristas sin dirección"]
print("Aporte incremental de cada relajación:\n")
display(delta[["Svc F1 medio", "Edge F1 medio", "MDE Svc n=30", "MDE Edge n=30"]].round(2))

est, perm = permisivo.iloc[0], permisivo.iloc[-1]
print(f"\nEstricto  -> permisivo completo:")
print(f"  Edge F1 medio : {est['Edge F1 medio']:.2f} -> {perm['Edge F1 medio']:.2f} "
      f"({perm['Edge F1 medio'] - est['Edge F1 medio']:+.2f})")
print(f"  MDE Edge      : {est['MDE Edge n=30']:.2f} -> {perm['MDE Edge n=30']:.2f} "
      f"({perm['MDE Edge n=30'] - est['MDE Edge n=30']:+.2f})")
razon = (perm['Edge F1 medio'] - est['Edge F1 medio']) / (est['MDE Edge n=30'] - perm['MDE Edge n=30']) \
    if est['MDE Edge n=30'] != perm['MDE Edge n=30'] else float("inf")
print(f"\n  Puntos de F1 ganados por cada punto de MDE reducido: {razon:.1f}")
print("  (un numero alto significa que la permisividad sube el puntaje mucho mas")
print("   de lo que mejora la capacidad de distinguir condiciones)")

Aporte incremental de cada relajación:



,Svc F1 medio,Edge F1 medio,MDE Svc n=30,MDE Edge n=30
+ agrupar actores,4.88,2.04,-0.04,0.09
+ aristas sin duplicar,0.00,3.29,0.00,-0.29
+ aristas sin dirección,0.00,10.97,0.00,-0.24



Estricto  -> permisivo completo:
  Edge F1 medio : 66.15 -> 82.44 (+16.30)
  MDE Edge      : 3.29 -> 2.85 (-0.44)

  Puntos de F1 ganados por cada punto de MDE reducido: 37.1
  (un numero alto significa que la permisividad sube el puntaje mucho mas
   de lo que mejora la capacidad de distinguir condiciones)


## 8 · El resultado, contra el efecto del oráculo

Con el piso de ruido calculado **solo con el panel de 30**, ya se puede decidir si el
salto del oráculo es real o cabe dentro del azar.

In [9]:
PROD = ["2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30",
        "2026-08-27_1640_STAGE2_V6_CORRECTED_cell9_p30_rep2",
        "2026-08-29_0442_STAGE2_V6_CORRECTED_cell9_p30_rep3"]
ORAC = ["2026-08-28_0315_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt",
        "2026-08-28_2211_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt_rep2"]


def promedio_condicion(dirs, metrica):
    idx = 0 if metrica == "svc" else 1
    por_video = {}
    for dirname in dirs:
        for v, tup in scores(load_run(dirname)).items():
            por_video.setdefault(v, []).append(tup[idx])
    return {v: float(np.mean(xs)) for v, xs in por_video.items()}


filas = []
for metrica, etiqueta in (("svc", "Service F1"), ("edge", "Edge F1")):
    prod = promedio_condicion(PROD, metrica)
    orac = promedio_condicion(ORAC, metrica)
    comunes = sorted(set(prod) & set(orac))
    p = np.mean([prod[v] for v in comunes])
    o = np.mean([orac[v] for v in comunes])
    mde = resumen_df.loc[etiqueta, "MDE n=30"]
    filas.append({
        "metrica": etiqueta,
        "produccion": p,
        "oraculo": o,
        "delta": o - p,
        "MDE n=30": mde,
        "veces el MDE": (o - p) / mde,
        "veredicto": "REAL" if abs(o - p) > mde else "dentro del ruido",
    })

print(f"Produccion promedia {len(PROD)} replicas, oraculo {len(ORAC)}; "
      f"ambas sobre los mismos 30 videos.\n")
display(pd.DataFrame(filas).set_index("metrica").round(2))

Produccion promedia 3 replicas, oraculo 2; ambas sobre los mismos 30 videos.



,produccion,oraculo,delta,MDE n=30,veces el MDE,veredicto
metrica,,,,,,
Service F1,87.43,99.57,12.14,1.76,6.88,REAL
Edge F1,58.22,82.17,23.95,3.29,7.28,REAL


In [10]:
# Test de signos: en cuantos videos gana el oraculo, sin suponer normalidad.
from math import comb

for metrica, etiqueta in (("svc", "Service F1"), ("edge", "Edge F1")):
    prod = promedio_condicion(PROD, metrica)
    orac = promedio_condicion(ORAC, metrica)
    comunes = sorted(set(prod) & set(orac))
    gana = sum(1 for v in comunes if orac[v] > prod[v])
    pierde = sum(1 for v in comunes if orac[v] < prod[v])
    empata = len(comunes) - gana - pierde
    n = gana + pierde
    p_val = min(1.0, 2 * sum(comb(n, k) for k in range(min(gana, pierde) + 1)) / 2**n) if n else 1.0
    print(f"{etiqueta}: oraculo gana en {gana}, pierde en {pierde}, empata en {empata} "
          f"(de {len(comunes)})  ->  p = {p_val:.5f}")

Service F1: oraculo gana en 25, pierde en 0, empata en 5 (de 30)  ->  p = 0.00000
Edge F1: oraculo gana en 28, pierde en 1, empata en 1 (de 30)  ->  p = 0.00000


## Nota sobre el alcance de este número

El `sigma_d` que sale acá está estimado con **2 pares** (60 observaciones), sobre el
panel de 30. Eso lo hace más específico que el número de `reports/noise_floor.json`
—que junta pares del panel de 14 y llega a ~9 pares— pero también más ruidoso como
estimación: un desvío estimado con pocos pares tiene su propio error.

Los dos coinciden en lo que importa para la conclusión: el salto del oráculo es
varias veces el MDE bajo cualquiera de las dos estimaciones, así que no depende de
cuál se elija.

Una advertencia sobre las réplicas del oráculo: son una réplica nula **válida**
(mismo prompt, misma entrada, dos llamadas), pero miden el ruido *bajo la condición
oráculo*. Que su sigma sea algo menor que el de producción es esperable — con una
entrada perfecta el modelo tiene menos margen para variar. Juntarlas con las de
producción da un promedio de ambos regímenes; si te interesa el ruido de producción
puro, usá solo el primer par.

## 9 · Exportación de las constantes

El notebook `02_evidencia_ablacion.ipynb` **carga** este archivo en vez de recalcular
el ruido. Así hay una sola fuente de verdad y el umbral queda fijado antes de mirar
las hipótesis.

> **Conflicto conocido:** `reports/noise_floor.json` reporta sigma_d de Edge = 8.76,
> calculado con ~9 pares que en su mayoría vienen del panel de 14. Este notebook da un
> valor distinto porque usa **solo** pares del panel de 30. Ambos son defendibles,
> pero no pueden convivir como número citable: **manda este**, porque el MDE tiene que
> corresponder al `n` que realmente se usa. El script viejo queda como verificación
> cruzada, no como fuente.

In [11]:
SALIDA = Path("../reports/piso_ruido_panel30.json")

constantes = {
    "generado": pd.Timestamp.utcnow().isoformat(),
    "fuente": "notebooks/01_calibracion_ruido.ipynb",
    "panel": 30,
    "Z_MDE": Z_MDE,
    "pares_usados": [{"condicion": p["condicion"], "run_a": p["run_a"], "run_b": p["run_b"]}
                     for p in pares],
    "n_observaciones": int(len(next(iter(todas.values())))),
    "metricas": {
        nombre: {
            "sigma_d": float(piso.loc[nombre, "sigma_d"]),
            "mde_14": float(piso.loc[nombre, "MDE n=14"]),
            "mde_30": float(piso.loc[nombre, "MDE n=30"]),
        }
        for nombre in piso.index
    },
    "protocolo_permisivo": {
        etiqueta: {
            "svc_f1_medio": float(permisivo.loc[etiqueta, "Svc F1 medio"]),
            "edge_f1_medio": float(permisivo.loc[etiqueta, "Edge F1 medio"]),
            "sigma_d_svc": float(permisivo.loc[etiqueta, "sigma_d Svc"]),
            "sigma_d_edge": float(permisivo.loc[etiqueta, "sigma_d Edge"]),
            "mde_svc_30": float(permisivo.loc[etiqueta, "MDE Svc n=30"]),
            "mde_edge_30": float(permisivo.loc[etiqueta, "MDE Edge n=30"]),
        }
        for etiqueta in permisivo.index
    },
    "nota": "Reemplaza a reports/noise_floor.json como fuente citable. Aquel usa "
            "pares del panel de 14 y da sigma_d Edge 8.76; este usa solo panel de 30. "
            "El bloque protocolo_permisivo calibra 4 niveles acumulativos de "
            "relajacion: cada uno es un instrumento distinto con su propio MDE.",
}

SALIDA.write_text(json.dumps(constantes, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"escrito -> {SALIDA}")
print(json.dumps(constantes["metricas"], indent=2, ensure_ascii=False))

escrito -> ../reports/piso_ruido_panel30.json
{
  "Edge F1": {
    "sigma_d": 6.431404305987118,
    "mde_14": 4.812822285765287,
    "mde_30": 3.2877835338366688
  },
  "Service F1": {
    "sigma_d": 3.4510388806363053,
    "mde_14": 2.582521023955381,
    "mde_30": 1.764197718968424
  },
  "aristas generadas": {
    "sigma_d": 1.4205553590374038,
    "mde_14": 1.063046290492722,
    "mde_30": 0.7261988667076605
  },
  "nodos generados": {
    "sigma_d": 0.663004925009435,
    "mde_14": 0.49614745502581103,
    "mde_30": 0.33893323628704264
  },
  "svc alucinados": {
    "sigma_d": 0.4144698643019561,
    "mde_14": 0.3101608458721214,
    "mde_30": 0.2118801944786674
  },
  "svc faltantes": {
    "sigma_d": 0.21081851067789198,
    "mde_14": 0.1577621275493231,
    "mde_30": 0.10777205024873014
  }
}


/tmp/ipykernel_66934/1206853037.py:4: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "generado": pd.Timestamp.utcnow().isoformat(),
